# ?? **Gemini + LangChain Essentials: LLM Calls, Prompts, Chains & RAG**

This notebook is a simple, beginner-focused guide to using **Gemini** and **LangChain**.
You?ll start with basic LLM calls and slowly build up to structured prompts, chains, and a small RAG example.
<br>
## ?? **What You Will Learn (Short & Friendly)**

* How to call Gemini & Groq models
* How to use Gemini inside LangChain
* How to work with prompts and output parsers
* How to build simple chains
* Basics of chunking, embeddings & retrieval (mini-RAG)

<br>

## ?? **Prerequisites**

* Basic Python
* A Google API key (Gemini)
* (Optional) OpenRouter + Groq key

<br>

## ?? **Goal of This Notebook**

To give you the **core building blocks** you need before moving into tools, memory, and agent workflows.

Let?s dive in!

<br>

## **Note**  
This notebook documents my learning journey.  
The implementations are practical and based on real issues I encountered.  
They may or may not be production-ready, but they are intended to help beginners understand the concepts clearly.



# **AI Agents**

Whenever you see a `!pip install <package>` command in this notebook, it simply means we are installing the required Python packages needed for that section to run.

In [ ]:
!pip install grandalf

##LLM calls

In [ ]:
!pip install -q -U google-genai

Gemini LLM Call

In [ ]:
#import API key from userdata

from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [ ]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Explain how AI works in a single line"
)
print(response.text)

Groq LLM Access through OpenRouter (OpenAI SDK)

In [ ]:
from google.colab import userdata
OPENROUTER_API_KEY=userdata.get('OPENROUTER_API_KEY')

In [ ]:
from openai import OpenAI
import json

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key= OPENROUTER_API_KEY,
)

completion = client.chat.completions.create(
  extra_headers={
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  model="x-ai/grok-4-fast:free",
  messages=[
    {
      "role": "user",
      "content": "What is the meaning of life?"
    }
  ]
)

# To convert response into dictionary object
# response_dict = completion.to_dict()
# pretty print of json with proper formatting
# print(json.dumps(response_dict, indent=2))
# print()

# normal response from oprn router api
# print(completion)

print(completion.choices[0].message.content)


Using Gemini with LangChain (Google GenAI Integration)

In [ ]:
!pip install langchain_google_genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import json

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",api_key=GOOGLE_API_KEY,)
response = llm.invoke("Write me a ballad about LangChain in 8 lines")
print(response.content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import json

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",api_key=GOOGLE_API_KEY,)
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Telugu. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
response = llm.invoke(messages)
print(response.content)

In [ ]:
!pip install langchain-core

In [ ]:
#Alternative way of Creating messages

from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

message=[
    SystemMessage(content="You are a helpful assistant that translates English to Telugu. Translate the user sentence."),
    HumanMessage(content="I love programming."),
    AIMessage(content="In Telugu : ???? ????????????? ???? ???? ????? \n (N?ku pr?gr?mi?g a??? c?l? i??a?.)."),
    HumanMessage(content="I am an AI/ML Engineer")
]
response = llm.invoke(message)
print(response.content)

## **Langchain Prompts**

### ?? What are Prompt Templates?

A Prompt Template is a **pre-structured prompt** where you can insert values later.  
Think of it like a form: you design the structure once and fill in the blanks whenever you need.


In [ ]:
!pip install langchain

### `ChatPromptTemplate` class

Using `from_template` to build a ChatPromptTemplate


In [ ]:
#langchain prompts

from langchain.prompts import ChatPromptTemplate

template = "You are a helpful assistant that translates {input_language} to {output_language} and the text to translate is : {text}."
prompt = ChatPromptTemplate.from_template(template)
prompt_value = prompt.invoke({"input_language":"English","output_language":"Hindi","text":"My friend is a Full stack developer"})

response = llm.invoke(prompt_value)
print(response.content)


In [ ]:
prompt_value = prompt.invoke({"input_language":"Telugu","output_language":"Hindi","text":"???? ?? AI/ML ???????"})
response = llm.invoke(prompt_value)
print(response.content)

Using `from_messages` to build a ChatPromptTemplate


In [ ]:
template_messages=[
    ("system", "You are a professional and helpful assistant that translates {input_language} to {output_language}. Translate the given user sentence."),
    ("human", "The text to translate is : {text}"),
]
prompt_template_with_messages = ChatPromptTemplate.from_messages(template_messages)
prompt_value = prompt_template_with_messages.invoke({"input_language":"English","output_language":"Hindi","text":"My friend is a Full stack developer"})
print(prompt_value)
# response = llm.invoke(prompt_value)
# print(response.content)

### `PromptTemplate` class

In [ ]:
from langchain_core.prompts import PromptTemplate

template = "Explain the topic {topic} in 5 simple lines to a {age}-year-old."
prompt = PromptTemplate(
    template = template,
    input_variables = ["topic","age"],
    partial_variables = {"age":"10"}

)
prompt_value = prompt.invoke({"topic":"AI"})
print(prompt_value)
response = llm.invoke(prompt_value)
print(response.content)

#chain method

# chain = prompt | llm
# response = chain.invoke({"topic":"AI"})
# print(response.content)

In [ ]:
# print(response)
# print(json.dumps(response.dict(),indent=2))

## **Schema-Guided Responses with `with_structured_output`**

### ?? Schema-Guided Responses (`with_structured_output`)

`with_structured_output` lets you tell the model **exactly what shape the output should follow**?like a form or schema.
Instead of free text, the model responds in a **structured format** (like JSON with specific fields).
This makes the output **predictable, clean, and easy to use in code.**




**TypedDict**

In [ ]:
from typing import TypedDict

class Person(TypedDict):
  name:str
  age: int
p1:Person = {"name":"John","age":30}
print(p1)
print(type(p1))

In [ ]:
from typing import TypedDict,Annotated,Literal

class Review(TypedDict):
  sentiment: Annotated[Literal['positive', 'neutral', 'negative'],"Overall sentiment of The review"]  # e.g., 'positive', 'neutral', 'negative'
  summary: str    # A concise summary of the review
  rating: float   # Numerical rating given by the reviewer
  rating_scale: str  # Scale used for the rating, e.g., '1-5', '0-10'

In [ ]:
review_input = """Karthik Gattamneni crafts an ambitious film that blends ancient mythology with modern superhero storytelling. The tale begins centuries ago, when Emperor Ashoka, shaken by the bloodshed of the Kalinga war, seals the secret of immortality within nine sacred books and entrusts them to loyal guardians.
Teja Sajja, following his success in Hanu-Man, delivers a solid performance as Vedha. Manchu Manoj shines as Mahabir Lama ? a character shaped by caste-based prejudice and societal rejection. His arc adds emotional complexity, blurring the lines between villainy and vengeance. Is he evil, or simply a man pushed to darkness by years of humiliation?
Visually, the film excels. The Sampati bird sequence and the high-octane train fight stand out as technical highlights, showcasing Gattamneni?s skill as a cinematographer. These moments elevate the film?s scale and immerse the audience in its mythic world.
Supporting performances by Jagapathi Babu, Jayaram and Shriya Saran add weight to Vedha?s journey. Ritika Nayak does well in a limited role as Vibha, while Gowra Hari?s background score enhances the epic tone.
However, the screenplay falters at times with unnecessary comedic detours that slow the pacing. The climax, though emotionally grounded, feels predictable and lacks the punch needed to elevate the finale.
Despite some narrative flaws, the film stands out for its visual grandeur, mythological depth, and strong performances ? making it a worthwhile watch, especially for fans of stylised, myth-inspired superhero tales."""

In [ ]:
structured_model = llm.with_structured_output(Review)
response1 = structured_model.invoke(review_input)
print(response1)

**Pydantic**

In [ ]:
from pydantic import BaseModel,Field
from typing import TypedDict,Annotated,Literal

class Person(BaseModel):
  name:str
  age:int

In [ ]:
p = {"name":"Viswa","age":16}
po=Person(**p)
print(po)

In [ ]:
class ReviewPydantic(BaseModel):
    sentiment: Annotated[
        Literal['positive', 'neutral', 'negative'],
        "Overall sentiment of the review"
    ]
    summary: Annotated[
        str,
        "A concise summary of the review"
    ]
    rating: Annotated[
        float,
        Field(..., ge=0, le=5, description="Numerical rating given by the reviewer (0 to 5)")
    ]

smodel_with_pydantic = llm.with_structured_output(ReviewPydantic)
response2 = smodel_with_pydantic.invoke(review_input)
print(response2)


**JSON**

In [ ]:
review_json_schema = {
  "title": "Generated schema for Review",
  "type": "object",
  "properties": {
    "summary": {
      "type": "string",
      "description": "A concise summary of the review" #Just like "Annotated" in JSON
    },
    "sentiment": {
      "type": "string",
      "enum" : ["hit","flop"], #Just like "Literal" in JSON
      "description": "Overall sentiment of the review"
    },
    "rating": {
      "type": "number",
      "description": "Numerical rating given by the reviewer (0 to 5)"
    },
    "rating_scale": {
      "type": "string",
      "enum": ["1-5","0-10"],
      "description": "Scale used for the rating, e.g., '1-5', '0-10'"
    }
  },
  "required": [
    "summary",
    "sentiment",
    "rating",
    "rating_scale"
  ]
}


# smodel_with_json = llm.with_structured_output(review_json_schema)
# response3 = smodel_with_json.invoke(review_input)
# print(response3)



#Gemini doesn't support JSON Mode

## **Output Parser**

### ?? What Are Output Parsers?

Output Parsers help convert the model?s raw text into a **clean, usable format**?like plain text, JSON, or a Python object.
They make LLM outputs easier to handle in pipelines and agent workflows.

<br>

# Short Explanation Per Parser

### **? `StrOutputParser`**

Converts the model?s answer into clean plain text.

### **? `JsonOutputParser`**

Parses the response into JSON (dictionary) you can use in code.

### **? `StructuredOutputParser`**

Enforces a more strict schema using JSON Schema.

### **? `PydanticOutputParser`**

Parses the output into a **typed Python class** (Pydantic model), great for agents.


**String Output Parser**

In [ ]:
import json
from langchain_core.output_parsers import StrOutputParser

parser =StrOutputParser()

chain = prompt | llm | parser
response = chain.invoke({"topic":"AI"})
print(response)

# print(json.dumps(response,indent=2))
# ---> json.dumps only knows how to serialize basic types (dict, list, str, int, etc.).
# ---> LangChain?s BaseMessage classes (like AIMessage, HumanMessage, SystemMessage) implement a .dict() method.
# print(response) // Raw ---> Formatted
# print(json.dumps(response.dict(),indent=2))

# TL;DR (I know you?re lazy ?? !!):
# ?? Without parser ? llm returns AIMessage (object with .content)
# ?? With StrOutputParser ? extracts .content ? plain str output ?
# ?? So response is string, no need .dict() or json.dumps()


**JSON Output Parser**

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

template = "Explain the topic {topic} in 5 simple lines to a {age}-year-old.Follow the format Instructions attached : {format_instructions}"
prompt = PromptTemplate(
    template = template,
    input_variables = ["topic","age"],
    partial_variables = {"age":"10","format_instructions":json_parser.get_format_instructions()}
)

chain = prompt | llm | json_parser
response = chain.invoke({"topic":"Indian Music","age":20})
print(response)

In [ ]:
#without chain for json parser
chain = prompt | llm
response = chain.invoke({"topic":"Western Music","age":20})

parsed_output = json_parser.invoke(response.content)
print(parsed_output)

parsed_output = json_parser.parse(response.content)
print(parsed_output)


**Structured Output Parser**

In [ ]:
from langchain.output_parsers import StructuredOutputParser,ResponseSchema

response_schemas = [
    ResponseSchema(name="topic", description="The explained topic in 5 simple lines"),
    ResponseSchema(name="summary", description="A 1-2 line overall summary of the topic"),
    ResponseSchema(name="analogy", description="A simple analogy or metaphor appropriate for the given age"),
    ResponseSchema(name="keywords", description="List of 3?5 important keywords from the topic"),
]

structured_output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
#prints instructions
print(structured_output_parser.get_format_instructions())




In [ ]:
template = "Explain the topic {topic} in 5 simple lines to a {age}-year-old.Follow the format Instructions attached : {format_instructions}"
prompt = PromptTemplate(
    template = template,
    input_variables = ["topic","age"],
    partial_variables = {"age":"10","format_instructions":structured_output_parser.get_format_instructions()}
)

chain = prompt | llm | structured_output_parser
response = chain.invoke({"topic":"Indian Music","age":20})
print(type(response))
print(json.dumps(response , indent=2))

In [ ]:
print(chain.get_graph().print_ascii())

**Pydantic Output Parser**

In [ ]:
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import List
from langchain_core.prompts import PromptTemplate

class TopicResolution(BaseModel):
    topic: str = Field(description="The name of the explained topic")
    summary: str = Field(description="A 1-2 line overall summary of the topic")
    analogy: str = Field(description="A simple analogy or metaphor appropriate for the given age")
    keywords: List[str] = Field(description="List of 3?5 important keywords from the topic")

pydantic_parser = PydanticOutputParser(pydantic_object=TopicResolution)
print(pydantic_parser.get_format_instructions())

In [ ]:
template = "Explain the topic {topic} in 5 simple lines to a {age}-year-old.Follow the format Instructions attached : {format_instructions}"
prompt = PromptTemplate(
    template = template,
    input_variables = ["topic","age"],
    partial_variables = {"age":"10","format_instructions":pydantic_parser.get_format_instructions()}
)

chain = prompt | llm | pydantic_parser
response = chain.invoke({"topic":"Indian Music","age":20})
print(type(response))
print(response)

In [ ]:
print(chain.get_graph().print_ascii())

## **Chains** (Contd..)

### Sequential Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize StrOutputParser
str_parser = StrOutputParser()

template_1 = """You are a professional linguist and translation specialist.
Translate the provided text from {input_language} into {output_language},
ensuring accuracy and preserving tone.

Translate the following review:
{review}

Provide only the translated review in {output_language}. Do not add explanations or commentary."""

template_2 = """
You are a professional content summarizer.
Summarize the provided reviewinto bullet points in the same language,
ensuring that the key points and important details are preserved,
while keeping the summary concise and easy to read.

Review to summarize:
{review}

Provide only the summarized review. Do not add explanations or commentary.
"""

prompt1 = PromptTemplate(
    template=template_1,
    input_variables=["input_language", "output_language", "review"]
)

prompt2 = PromptTemplate(
    template=template_2,
    input_variables=["review"]
)

review_input = """
As someone who grew up with the original Lilo & Stitch, I wasn't sure what to expect from the 2025 remake-but wow, this movie absolutely delivered. It's heartfelt, beautifully made, and captures the spirit of the original while adding just the right amount of freshness.

This is the kind of remake we actually want-one made with care, love, and deep respect for the source material. The emotional themes about family, love, and belonging are still front and center, and they hit just as hard today as they did years ago. Lilo's story is just as touching, Stitch is just as chaotic and adorable, and the message of "ohana" still resonates powerfully.

Visually, the movie is stunning. The Hawaiian setting feels authentic and vibrant, and the design of Stitch and the other alien characters blends surprisingly well with the real-world elements. It never feels forced or overly modernized. The music also keeps that nostalgic charm with new energy.

I could tell the creators worked hard on this. It's not a cash grab-it's a genuine love letter to fans old and new. Whether you've seen the original or not, this remake stands tall on its own.

This is probably the best Disney remake so far. 10/10. It's emotional, entertaining, and full of heart. A true example of how remakes should be.

"""

chain_1 = prompt1 | llm | str_parser
chain_2 = prompt2 | llm | str_parser

final_chain = chain_1 | chain_2
response = final_chain.invoke({"input_language":"English","output_language":"Telugu","review":review_input})
print(response)


In [ ]:
print(final_chain.get_graph().print_ascii())

**Note:**

The setup works only because `prompt2` is a `PromptTemplate` with a single input variable. Since we use `StrOutputParser`, the previous chain produces a single output, which LangChain can automatically map to the single input of `prompt2` under the hood.

If `prompt2` required more than one input variable, LangChain would fail to map the single output to multiple inputs and would raise a `TypeError`.


### Parallel Chains

In [ ]:
review_input = """I was waiting for the release of the dubbed version of Doraemon's this movie since a long time and now that it finally released, i couldn't hold myself from watching it.

And it was just the bestest movie i would say I've watched. Waiting worths it all as it was not only fun-filled but also conveyed a very important message which I deeply understood and would want you guys to understand too.

So here's, How It Taught Me the Value of Imperfections:

I've always believed that Doraemon movies have a special kind of magic. They never fail to entertain, to touch my heart, and to leave me with something meaningful. But this movie is different. It's not just a magical adventure or a fun journey in the skies which gave me entertainment but gave a lesson.

Nobita's Sky Utopia was different. It wasn't just another movie; it was a lesson in life-a lesson that I didn't realize I needed until I watched it. It gave a profound reminder that being human means being imperfect, and that our flaws aren't weaknesses-they're what connect us to others, to feelings, to empathy, to love.

What this film portrayed so powerfully is the beauty of human imperfections. It made me realize that our flaws-the ones we often try to hide or hate-are actually essential. They're what make us human.

Our hearts, with all their chaos, pain, and softness, are what guide us through life more than our logic ever can. The message that perfection might look beautiful, but it's emotionless, mechanical, and unfeeling-that was portrayed so powerfully in this movie. Without flaws, we wouldn't feel love, compassion, growth, or understanding. Without conflict, there would be no evolution-within ourselves or in society. Without them, we wouldn't be able to love deeply, feel compassion, understand pain, or truly connect with others. Perfection might look ideal, but it's lifeless. It's our imperfections that make our hearts alive.

The message that every decision doesn't have to be logical-that our heart, with all its messiness, matters too-was so clearly and beautifully shown. I've been struggling lately with my own flaws, with conflict in my family, and with painful thoughts about myself and the world. But this movie brought me hope. It reminded me that maybe we're not supposed to be perfect. Maybe we're just meant to try, to learn, and to grow through our mistakes and feelings.

There's a part in the movie where Nobita sees his friends-Shizuka, Gian, and Suneo-change under Dr. Ray's manipulation. At first, the change seems ideal: they're no longer fighting, no more teasing, no more chaos. It looks perfect. But as time passes, Nobita realizes that he misses their flaws-their imperfections, the very things that used to annoy him. And it made me realize something so powerful: sometimes, the flaws that irritate us the most are the very things that make those people unforgettable.

It's so easy to get frustrated with the imperfections of those around us, but when they're gone, we realize that those imperfections are the very traits that make us love them. The quirks, the annoyances, the little things that make someone who they are. These flaws aren't just mistakes; they're parts of the magic of being human.

So, we should try to handle our emotions before pointing the flaws of others because once they're lost, we might realize their value.

This movie brought me back to a memory of a friend I have-a friend who used to be so talkative, so obsessed with being around me. Back then, it felt overwhelming at times, even irritating. But now, something's changed. She's quieter. And honestly, I miss the version of her who couldn't stop talking, who would fill up the space with her energy and love. I didn't realize it then, but her flaws were what made her special to me.

This movie reminded me that imperfections are necessary. They aren't something to be ashamed of. In fact, they are what make our relationships real and meaningful. If everything was perfect, we wouldn't be able to feel the love, the connection, the depth that comes with imperfection.

It helped me want to live. To survive. To embrace life with all its pain and beauty.

So, to the creators of Nobita's Sky Utopia, thank you for bringing this beautiful lesson to life.

Thank you, from the deepest corner of my heart, for creating such a powerful film. It's a story that will stay with me forever. It made me appreciate the people in my life, flaws and all, and reminded me to embrace the chaos and imperfections that come with being human. Sometimes, it's those very flaws that connect us in the most profound way.

I truly encourage everyone to watch Nobita's Sky Utopia. It's not just a Doraemon adventure-it's a lesson in being human."""

In [ ]:
template_1 = """You are a professional linguist and translation specialist.
Translate the provided text from {input_language} into {output_language},
ensuring accuracy and preserving tone.

Translate the following review:
{review}

Provide only the translated review in {output_language}. Do not add explanations or commentary."""

template_2 = """You are a professional content curator and social media specialist.
Generate a set of relevant and engaging hashtags based on the provided review,
ensuring they reflect its key themes, sentiment, and tone.

Review:
{review}

Provide only the hashtags, separated by spaces. Do not add explanations or commentary."""

template_3 = """You are a professional content strategist.
Summarize the provided review in the same language and also in a concise and engaging way, keeping its key sentiment and tone.
Do not modify or add to the provided hashtags ? keep them exactly as they are.

Review:
{translated_review}

Hashtags:
{hashtags}

Output format:
Summary: <summarized review>
Hashtags: {hashtags}

Do not add explanations or commentary."""

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel

parser = StrOutputParser()

prompt1 = PromptTemplate(
    template=template_1,
    input_variables=["input_language", "output_language", "review"]
)
prompt2 = PromptTemplate(
    template=template_2,
    input_variables=["review"]
)
prompt3 = PromptTemplate(
    template=template_3,
    input_variables=["translated_review", "hashtags"]
)

parallel_chain = RunnableParallel({
  "translated_review":prompt1 | llm | parser,
    "hashtags":prompt2 | llm | parser,
})

sequentiual_chain = prompt3 | llm | parser
final_chain = parallel_chain | sequentiual_chain
response = final_chain.invoke({"input_language":"English","output_language":"Telugu","review":review_input})
print(response)


In [ ]:
#Output will be:

# Summary: ???? ?????? ?????????????? ???????? ??????, "?????? ???? ????????" ???????? ??????, ??? ????? ??????? ???????, ???? ?????????? ???????, ?????? ?????? ????????? ??? ???????? ????????? ????? ????? ???????? ?????????. ????????? ???? ?????, ????????????, ???????, ????? ??????? ????? ??????????? ?????? ?????? ??????????, ?? ????????? ???????? ????????? ? ?????? ??????????? ???????????????. ??? ???? ????????, ?????????? ?????, ??????? ???????????????, ?? ?????? ???????? ??????? ???? ??????????????? ???????????????. ????? ?????? ????? ????????? ??????, ??? ????????? ??????? ?? ????????? ????.
# Hashtags: #Doraemon #NobitasSkyUtopia #MovieReview #Anime #JapaneseAnimation #LessonLearned #Imperfection #EmbraceYourFlaws #Humanity #Empathy #Love #Connection #HeartOverLogic #SelfAcceptance #GrowthMindset #MeaningfulMovie #MustWatch #DoraemonMovie #Nostalgia #ChildhoodMemories #EmotionalJourney #RealConnections #AppreciateWhatYouHave

In [ ]:
print(final_chain.get_graph().print_ascii())

## **RAG Introduction**

### **Document Loaders**

**Document Loader**: In LangChain, a Document Loader is a component that **reads data from a source (like PDFs, websites, or databases) and converts it into a standardized document format that can be processed by the framework**.


**Text Loader**

In [ ]:
!pip install langchain_community

In [ ]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader('/content/Thompson.txt')
loaded_doc = text_loader.load()
print(loaded_doc)
print(type(loaded_doc))
print(type(loaded_doc[0]))


**PDF loader**

In [ ]:
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdfLoader = PyPDFLoader('/Pravaha_pvt.pdf')
loaded_doc = pdfLoader.load()
print(len(loaded_doc))
print(loaded_doc[4])

**CSV loader**

In [ ]:
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader("/content/earthquake_alert_balanced_dataset.csv")
loaded_csv = csv_loader.load()
print(type(loaded_csv))
print(len(loaded_csv))
print(loaded_csv[0])

**Web Loader**

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader('https://narasimhakambham.netlify.app/')
loaded_web_doc = web_loader.load()
print(len(loaded_web_doc))

In [ ]:
print(loaded_web_doc[0])

**Directory Loader**

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

# Load TXT files
txt_loader = DirectoryLoader(
    path="/content/ex",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)
txt_docs = txt_loader.load()

# Load PDF files
pdf_loader = DirectoryLoader(
    path="/content/ex",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)
pdf_docs = pdf_loader.load()

# Combine documents
docs = txt_docs + pdf_docs
print(len(docs))
print("hello")


In [ ]:
#Alternate way of Directory loading

from langchain_community.document_loaders import TextLoader, PyPDFLoader
import glob, os

def load_docs(path="/content/ex"):
    docs = []
    # Get all files recursively
    files = glob.glob(os.path.join(path, "**/*"), recursive=True)

    for f in files:
        if os.path.isfile(f):
            if f.endswith(".txt"):
                loader = TextLoader(f)
                docs.extend(loader.load())
            elif f.endswith(".pdf"):
                loader = PyPDFLoader(f)
                docs.extend(loader.load())
            else:
                print(f"Skipping unsupported file type: {f}")
    return docs

docs = load_docs()
print(f"Total documents loaded: {len(docs)}")


### **Eager Loading VS Lazy Loading**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdfLoader = PyPDFLoader("""/content/Aur?lien G?ron - Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow_ Concepts, Tools, and Techniques to Build Intelligent Systems-O'Reilly Media (2022).pdf""")


In [ ]:
loaded_doc = pdfLoader.load()
print(type(loaded_doc))
print(len(loaded_doc))
print(loaded_doc[40])

In [ ]:
# Create a lazy-loaded PDF generator (does NOT load entire PDF into memory)
lazy_loaded_pdf = pdfLoader.lazy_load()
print(type(lazy_loaded_pdf)) # <class 'generator'> confirms it's a generator
for i, doc in enumerate(lazy_loaded_pdf):
  if i == 70 :
    print(type(doc))
    print(doc)
    break


###**Chunking/Splitting**

CharacterTextSplitter

In [ ]:
text ="""I had remarked during my stay in the United States that a democratic state of society, similar to that of the Americans, might offer singular facilities for the establishment of despotism; and I perceived, upon my return to Europe, how much use had already been made, by most of our rulers, of the notions, the sentiments, and the wants created by this same social condition, for the purpose of extending the circle of their power. This led me to think that the nations of Christendom would perhaps eventually undergo some oppression like that which hung over several of the nations of the ancient world.
A more accurate examination of the subject, and five years of further meditation, have not diminished my fears, but have changed their object.
No sovereign ever lived in former ages so absolute or so powerful as to undertake to administer by his own agency, and without the assistance of intermediate powers, all the parts of a great empire; none ever attempted to subject all his subjects indiscriminately to strict uniformity of regulation and personally to tutor and direct every member of the community. The notion of such an undertaking never occurred to the human mind; and if any man had conceived it, the want of information, the imperfection of the administrative system, and, above all, the natural obstacles caused by the inequality of conditions would speedily have checked the execution of so vast a design."""

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0 , separator ='')

In [ ]:
result = splitter.split_text(text)
print(type(result))
print(result)
print(result[0])

RecursiveCharacterTextSplitter

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=50,chunk_overlap=0,separators=['\n\n','\n',' ',''])

In [ ]:
rec_result = recursive_splitter.split_text(text)
print(type(rec_result))
print(rec_result)
print(rec_result[0])

In [ ]:
code_sample = """
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    seq = [0, 1]
    for i in range(2, n):
        seq.append(seq[-1] + seq[-2])
    return seq

class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

# Main block
if __name__ == "__main__":
    print("Fibonacci(10):", fibonacci(10))
    calc = Calculator()
    print("Add:", calc.add(5, 3))
    print("Multiply:", calc.multiply(5, 3))
"""

In [ ]:
#RecursiveCharacterTextSplitter for codeing languages

code_splitter = RecursiveCharacterTextSplitter.from_language(language="python",chunk_size=10,chunk_overlap=0)
response_for_code = code_splitter.split_text(code_sample)
print(len(response_for_code))
print(response_for_code[:10])


# ? Summary & Next Steps

In this notebook, you explored the **foundations of working with LLMs and LangChain using Gemini**:

### ?? You have done:

- Called **Gemini** directly using the `google-genai` client  
- Used **Groq** through **OpenRouter** with the OpenAI SDK  
- Integrated **Gemini with LangChain** via `ChatGoogleGenerativeAI`  
- Built prompts using:
  - `ChatPromptTemplate`
  - `PromptTemplate` with variables and partials  
- Parsed model outputs using:
  - `StrOutputParser` (plain text)
  - `JsonOutputParser`
  - `StructuredOutputParser`
  - `PydanticOutputParser` + `BaseModel`
- Created:
  - Simple chains (`prompt | llm`)
  - **Sequential chains** (translate ? summarize)
  - **Parallel chains** (translate + hashtags ? final summary)
- Touched **RAG basics**:
  - Text splitting (character & recursive)
  - Code splitting

<br>

## ?? What this notebook gives you

You now have the core building blocks for:

- Building **structured, predictable LLM workflows**
- Designing **translation, summarization, and content pipelines**
- Moving towards **full RAG systems** and **Agentic AI workflows**

This notebook is intentionally designed as a **foundation**.  
Next notebooks in the series will go deeper into:

- Tools & function calling  
- Memory and context management  
- Full RAG pipelines (query ? retrieve ? synthesize)  
- Multi-step **Agentic AI** workflows

<br>

?? **Tip:**  
If you found a section especially useful (e.g., output parsers or chains), copy that part into a separate mini-notebook and adapt it for your own project or domain.

**Keep experimenting. This is where real understanding starts. ??**
